# Capitolo 6 — Un Transformer minimo su Pinocchio (§ 6.9)
Stessa taglia della LSTM (828 000 parametri), 40 epoche (~20 minuti su 4 core). Riproduce il confronto 1.62 contro 1.37 di validazione.

In [ ]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import dati
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
fissa_seme(42)

import time
testo = dati.pinocchio(); caratteri = sorted(set(testo)); V = len(caratteri); c2i = {c: i for i, c in enumerate(caratteri)}
dati_t = torch.tensor([c2i[c] for c in testo], dtype=torch.long); n_tr = int(len(dati_t) * 0.9); tr, va = dati_t[:n_tr], dati_t[n_tr:]
L, B = 128, 64
def batch(d):
    ix = torch.randint(0, len(d) - L - 1, (B,)); return torch.stack([d[i:i + L] for i in ix]), torch.stack([d[i + 1:i + L + 1] for i in ix])

In [ ]:
class CharTransformer(nn.Module):
    def __init__(self, V, d=128, teste=4, strati=4, L=128):
        super().__init__()
        self.emb = nn.Embedding(V, d); self.pos = nn.Embedding(L, d)
        strato = nn.TransformerEncoderLayer(d, teste, dim_feedforward=4 * d, dropout=0.2, batch_first=True)
        self.enc = nn.TransformerEncoder(strato, strati); self.fc = nn.Linear(d, V)
    def forward(self, x):
        n = x.shape[1]; maschera = torch.triu(torch.full((n, n), float("-inf")), diagonal=1)
        return self.fc(self.enc(self.emb(x) + self.pos(torch.arange(n)), mask=maschera, is_causal=True))
fissa_seme(42); modello = CharTransformer(V); print("parametri:", sum(p.numel() for p in modello.parameters()))
perdita_fn = nn.CrossEntropyLoss(); opt = torch.optim.Adam(modello.parameters(), lr=1e-3); passi = len(tr) // (B * L); t0 = time.time()
for epoca in range(40):
    modello.train(); s = 0
    for _ in range(passi):
        x, y = batch(tr); opt.zero_grad(); perdita = perdita_fn(modello(x).reshape(-1, V), y.reshape(-1)); perdita.backward(); nn.utils.clip_grad_norm_(modello.parameters(), 1.0); opt.step(); s += perdita.item()
    modello.eval()
    with torch.no_grad(): x, y = batch(va); lv = perdita_fn(modello(x).reshape(-1, V), y.reshape(-1)).item()
    print(f"Epoca {epoca+1:2d}: train {s/passi:.3f} | val {lv:.3f} ({time.time()-t0:.0f} s)")